# 🧭 Nextify Course Notebook - Agent 1 (Idea Intake + Brainstorming)

This notebook follows the course practice flow for the **idea track**. It sets up the environment, collects the Idea Form, wires the Brainstorming agent (with sub-agents/tools), and produces a **Product Snapshot**. It uses the updated architecture with a cross-cutting **LLM Switch Agent** that serves every agent.


## 0. Environment setup (Kaggle-ready)
- Gemini is used first; when a token limit or error occurs, the Switch Agent can (with user confirmation) retry on OpenAI and carry forward summaries + outputs.
- Expects `GEMINI_API_KEY` and optionally `OPENAI_API_KEY` in the environment.
- Run the pip cell once per session if the libraries are missing.


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q google-generativeai openai python-dotenv


In [ ]:
import os, json, textwrap
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

# If you're on Kaggle, set API keys via the sidebar or environment.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GEMINI_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not GEMINI_API_KEY and not OPENAI_API_KEY:
    raise RuntimeError("Provide at least one API key: GEMINI_API_KEY or OPENAI_API_KEY")


## 1. Idea Form (user-filled)
Edit the placeholders to reflect your idea. This mirrors the course template.


In [ ]:
idea_form = {
    'idea_name': 'Nextify - Product Strategy Copilot',
    'persona': 'Founder building AI-first PM toolkit',
    'problem': 'Quickly turn raw ideas into prioritized product snapshots without context loss.',
    'target_user': 'Product managers and founders shipping MVPs',
    'success_outcome': 'A crisp snapshot (problem, users, solution sketch, risks) approved by user.',
    'constraints': ['Single journey: idea track', 'LLM-first, with guardrails', 'Time-box to one agent for now'],
}
print(json.dumps(idea_form, indent=2))


## 2. LLM Switch Agent (Gemini -> OpenAI) with user confirmation
- Serves all agents as a shared service.
- Tries Gemini first (course default). If a limit/error occurs, it **asks for subscriber confirmation** (set `user_allows_switch=True`) before retrying on OpenAI.
- Passes a compact continuity summary **plus prior outputs** (research, risks, user feedback) to the fallback provider.
- Returns both the response text and the provider actually used.


In [ ]:
class LLMError(Exception):
    ...

@dataclass
class LLMConfig:
    primary: str = 'gemini'  # 'gemini' or 'openai'
    gemini_model: str = 'gemini-1.5-pro-latest'
    openai_model: str = 'gpt-4o-mini'

class LLMSwitchingAgent:
    """Routes between Gemini and OpenAI, carrying continuity + prior outputs."""

    def __init__(self, cfg: Optional[LLMConfig] = None, max_trace_chars: int = 1800):
        self.cfg = cfg or LLMConfig()
        self.max_trace_chars = max_trace_chars

    def _call_gemini(self, user_prompt: str, system_prompt: Optional[str]) -> str:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel(self.cfg.gemini_model)
        resp = model.generate_content([p for p in [system_prompt, user_prompt] if p])
        if not resp or not resp.text:
            raise LLMError('Empty response from Gemini')
        return resp.text

    def _call_openai(self, user_prompt: str, system_prompt: Optional[str]) -> str:
        import openai
        if not OPENAI_API_KEY:
            raise LLMError('OpenAI key missing for fallback')
        client = openai.OpenAI(api_key=OPENAI_API_KEY)
        messages = []
        if system_prompt:
            messages.append({'role': 'system', 'content': system_prompt})
        messages.append({'role': 'user', 'content': user_prompt})
        resp = client.chat.completions.create(model=self.cfg.openai_model, messages=messages)
        return resp.choices[0].message.content

    def __call__(
        self,
        user_prompt: str,
        *,
        system_prompt: Optional[str] = None,
        continuity_summary: str = '',
        prior_outputs: Optional[List[str]] = None,
        user_allows_switch: bool = True,
        simulate_gemini_limit: bool = False,
    ) -> Tuple[str, str]:
        prior_outputs = prior_outputs or []
        trace = (continuity_summary + '
' + '
'.join(prior_outputs))[-self.max_trace_chars :]
        provider_order = [self.cfg.primary, 'openai' if self.cfg.primary == 'gemini' else 'gemini']
        for idx, provider in enumerate(provider_order):
            try:
                if provider == 'gemini':
                    if simulate_gemini_limit:
                        raise LLMError('Simulated Gemini token limit hit')
                    return self._call_gemini(user_prompt, system_prompt), 'gemini'
                else:
                    return self._call_openai(user_prompt, system_prompt), 'openai'
            except Exception as exc:
                if provider == 'gemini' and user_allows_switch and idx == 0:
                    fallback_prompt = textwrap.dedent(f"""
                        The primary provider failed due to: {exc}
                        Here is the continuity summary + prior outputs to continue seamlessly:
                        {trace}
                        Continue the task without repeating already completed steps.
                    """)
                    user_prompt = fallback_prompt + '

' + user_prompt
                    continue
                raise

switching_agent = LLMSwitchingAgent()


## 3. Brainstorming sub-agents (market analysis + crazy ideas + synthesis)
- **MarketAnalysisSubAgent**: builds TAM/SAM/SOM-style overview and market signals for the exact idea.
- **CrazyIdeaSubAgent**: remixes industries/technical trends to propose bold variants (ignores constraints to spark creativity).
- **SynthesisSubAgent**: merges both outputs, scores candidates, and drafts a Product Snapshot (extended Idea Form).


In [ ]:
def market_analysis_prompt(idea: Dict[str, Any]) -> str:
    return textwrap.dedent(f"""
    Act as MarketAnalysisSubAgent.
    Build a concise TAM/SAM/SOM summary for: {idea.get('idea_name')}.
    Include: target segments, pricing/ARPU assumption, payback guess, 3 risks, and 3 validation signals.
    Return a short markdown section.
    """)

def crazy_idea_prompt(idea: Dict[str, Any]) -> str:
    return textwrap.dedent(f"""
    Act as CrazyIdeaSubAgent.
    Remix the idea by combining 2-3 adjacent industries and emerging tech trends.
    Propose 3 bold concepts with 1-line rationale each. Ignore typical constraints to maximize originality.
    """)

def synthesis_prompt(idea: Dict[str, Any], market: str, crazy: str, user_feedback: str, revision_count: int) -> str:
    return textwrap.dedent(f"""
    Act as SynthesisSubAgent.
    Inputs:
    - Idea Form: {json.dumps(idea)}
    - Market analysis:
    {market}
    - Crazy concepts:
    {crazy}
    - User feedback: {user_feedback or 'n/a'}
    - Revision count so far: {revision_count} (maximum 2 revisions).

    Tasks:
    1) Merge the insights into a markdown table of candidates with columns: id, concept, TAM/SAM/SOM fit, feasibility (1-5), differentiation, risks, user value.
    2) Highlight the top pick and why.
    3) Draft a Product Snapshot (extended Idea Form) with fields: problem, users, proposed solution, market sizing note, pros, cons, risks, success metric, and 3 next validation steps.
    Keep it concise.
    """)


## 4. Brainstorming agent orchestration (short-term summary + switching)
- Runs the three sub-agents via the Switch Agent.
- Maintains a lightweight summary so provider switches keep context without long prompts.
- Supports up to **two revisions**; set `user_feedback` and re-run the loop.


In [ ]:
def run_brainstorm_cycle(
    idea: Dict[str, Any],
    *,
    user_feedback: str = '',
    continuity_summary: str = '',
    user_allows_switch: bool = True,
    revision_count: int = 0,
    simulate_gemini_limit: bool = False,
    cfg: Optional[LLMConfig] = None,
) -> Tuple[str, List[str], str, int]:
    # Refresh agent if cfg changes mid-run
    local_agent = switching_agent if cfg is None else LLMSwitchingAgent(cfg=cfg)
    providers_used: List[str] = []

    market_prompt = market_analysis_prompt(idea)
    market_analysis, provider_market = local_agent(
        market_prompt,
        system_prompt='Be precise and concise.',
        continuity_summary=continuity_summary,
        user_allows_switch=user_allows_switch,
        simulate_gemini_limit=simulate_gemini_limit,
    )
    providers_used.append(provider_market)

    crazy_prompt = crazy_idea_prompt(idea)
    crazy_output, provider_crazy = local_agent(
        crazy_prompt,
        system_prompt='Max creativity, still actionable.',
        continuity_summary=continuity_summary + '
' + market_analysis,
        user_allows_switch=user_allows_switch,
        simulate_gemini_limit=simulate_gemini_limit,
    )
    providers_used.append(provider_crazy)

    synth_prompt = synthesis_prompt(idea, market_analysis, crazy_output, user_feedback, revision_count)
    product_snapshot, provider_synth = local_agent(
        synth_prompt,
        system_prompt='Keep outputs structured, truthful, and concise.',
        continuity_summary=continuity_summary + '
' + market_analysis + '
' + crazy_output,
        user_allows_switch=user_allows_switch,
        simulate_gemini_limit=simulate_gemini_limit,
    )
    providers_used.append(provider_synth)

    updated_summary = textwrap.shorten(
        f"Market:
{market_analysis}
Crazy:
{crazy_output}
Snapshot:
{product_snapshot}
Feedback:{user_feedback}",
        width=1800,
        placeholder='...',
    )
    return product_snapshot, providers_used, updated_summary, revision_count

# Keep a shared summary state between iterations
summary_state = ''
revision_count = 0


## 5. First pass (no feedback yet)
Run this cell to generate the initial Product Snapshot. If Gemini hits a limit, it will ask for confirmation to switch to OpenAI and note the provider(s).


In [ ]:
cfg = LLMConfig(primary='gemini')  # flip to 'openai' if you want OpenAI first
product_snapshot, providers, summary_state, revision_count = run_brainstorm_cycle(
    idea_form, cfg=cfg, user_allows_switch=True
)
print(f"Providers used (market / crazy / synthesis): {providers}
")
print(product_snapshot)
print('
Continuity summary for potential switch:')
print(summary_state)


## 6. User confirmation loop (max 2 revisions)
1. Add your feedback to `user_feedback` (e.g., choose a candidate id, add constraints, or request tweaks).
2. Re-run the cell to iterate. Up to two revisions are supported.
3. The continuity summary keeps context tight for potential provider switches.


In [ ]:
user_feedback = ''  # <-- edit me, e.g., 'Pick concept C2, emphasize onboarding risk'
revision_count = min(revision_count + 1, 2)
product_snapshot, providers, summary_state, revision_count = run_brainstorm_cycle(
    idea_form,
    user_feedback=user_feedback,
    continuity_summary=summary_state,
    user_allows_switch=True,
    revision_count=revision_count,
    cfg=cfg,
)
print(f"Providers used (market / crazy / synthesis): {providers}
")
print(product_snapshot)
print('
Continuity summary for potential switch:')
print(summary_state)
